In [ ]:
# Setup and Functions
import requests
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import ai

API_KEY = "your_api_key_here" # IMPORTANT: Change this to your free API key from https://site.financialmodelingprep.com/


def fmp_get(endpoint, symbol):
    url = f"https://financialmodelingprep.com/stable/{endpoint}?symbol={symbol}&period=annual&apikey={API_KEY}"
    response = requests.get(url)
    if response.json():
        return response.json()[0]
    return {}

def analyze_company(symbol):
    profile = fmp_get("profile", symbol)
    metrics = fmp_get("key-metrics", symbol)
    financials = fmp_get("income-statement", symbol)

    if not profile or not metrics or not financials:
        print(f"Could not retrieve full data for {symbol}. Skipping.")
        return None

    revenue = financials.get("revenue")
    gross_profit = financials.get("grossProfit", 0)
    operating_income = financials.get("operatingIncome", 0)
    net_income = financials.get("netIncome", 0)

    return {
        "Company": profile.get("companyName"),
        "Symbol": symbol,
        "Sector": profile.get("sector"),
        "Price": profile.get("price"),
        "Market Cap": metrics.get("marketCap"),
        "Revenue": revenue,
        "Net Income": net_income,
        "EPS": financials.get("epsDiluted"),
        "Gross Margin %": round(gross_profit / revenue * 100, 2) if revenue else 0,
        "Operating Margin %": round(operating_income / revenue * 100, 2) if revenue else 0,
        "Net Margin %": round(net_income / revenue * 100, 2) if revenue else 0,
        "ROE": metrics.get("returnOnEquity"),
        "ROIC": metrics.get("returnOnInvestedCapital"),
        "EV/EBITDA": metrics.get("evToEBITDA"),
        "Earnings Yield": metrics.get("earningsYield"),
        "Current Ratio": metrics.get("currentRatio"),
        "Net Debt/EBITDA": metrics.get("netDebtToEBITDA"),
    }

In [ ]:
def full_analysis(companies):
    # 1. Pull data
    print("Pulling financial data...")
    rows = [analyze_company(s) for s in companies]
    rows = [r for r in rows if r is not None]
    df = pd.DataFrame(rows)

    # 2. Display table
    print("\n=== Financial Comparison ===\n")
    display(df)

    # 3. Charts
    categories = ["Gross Margin %", "Operating Margin %", "Net Margin %"]
    x = range(len(categories))
    width = 0.25

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Margins chart
    for i, row in df.iterrows():
        values = [row[cat] for cat in categories]
        offset = (i - 1) * width
        axes[0].bar([p + offset for p in x], values, width, label=row["Symbol"])
    axes[0].set_ylabel("Percentage (%)")
    axes[0].set_title("Profit Margins")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(categories)
    axes[0].legend()
    axes[0].axhline(y=0, color="black", linewidth=0.5)

    # Return on capital chart
    metrics = ["ROE", "ROIC"]
    x2 = range(len(metrics))
    for i, row in df.iterrows():
        values = [row[m] * 100 for m in metrics]
        offset = (i - 1) * width
        axes[1].bar([p + offset for p in x2], values, width, label=row["Symbol"])
    axes[1].set_ylabel("Percentage (%)")
    axes[1].set_title("Return on Capital")
    axes[1].set_xticks(x2)
    axes[1].set_xticklabels(metrics)
    axes[1].legend()
    axes[1].axhline(y=0, color="black", linewidth=0.5)

    # Debt chart
    symbols = df["Symbol"].tolist()
    debt_values = df["Net Debt/EBITDA"].tolist()
    colors = ["green" if v < 3 else "orange" if v < 7 else "red" for v in debt_values]
    axes[2].bar(symbols, debt_values, color=colors)
    axes[2].set_ylabel("Net Debt / EBITDA")
    axes[2].set_title("Debt Burden (Lower is Better)")
    axes[2].axhline(y=3, color="orange", linewidth=0.5, linestyle="--")
    axes[2].axhline(y=7, color="red", linewidth=0.5, linestyle="--")
    axes[2].axhline(y=0, color="black", linewidth=0.5)

    plt.tight_layout()
    plt.show()

    # 4. LLM Analysis
    print("\n=== AI Analysis ===\n")
    data_text = df.to_string()

    prompt = f"""You are a financial analyst. Here is financial data for several companies:

{data_text}

Weight these factors in order of importance:
1. Profitability (margins, ROE, ROIC) - most important
2. Financial health (debt levels, current ratio) - second
3. Valuation (EV/EBITDA, earnings yield) - third

A highly profitable company with high debt is riskier than a moderately profitable company with no debt. A cheap stock that is unprofitable is not a good value.

Output format:
- One short paragraph per company summarizing its overall picture (strengths, weaknesses, red flags) with specific numbers inline
- Then a final ranking section that directly compares the companies against each other, explaining why #1 beats #2 and #2 beats #3
- Be concise. No category-by-category breakdown per company.
- Present numbers as percentages where appropriate (e.g., 29.65% not 0.2965)
- No letter formatting, greetings, or dates"""

    response = ai.generate_text(prompt)
    print(response)

In [ ]:
# Run it with any stock tickers
full_analysis(["AAPL","F","MSFT"])